# 第二级

In [1]:
!pip install diskcache

Looking in indexes: https://mirrors.ustc.edu.cn/pypi/web/simple


In [2]:
# 先写一个加载器

class DataSet(object):
    def __init__(dataset_path):
        pass

In [3]:
# 由上到下，先看看怎么使用

class Histgraph(object):
    def __init__(dataset):
        pass

In [4]:
# 过程代码
import json,time
from tqdm import tqdm
from diskcache import Cache

class DataSet(object):
    def __init__(self, dataset_path='/data/dataset/wiki_zh'):
        self.max = 100000000
        self._cache = Cache('/tmp/nlp/')
        self._data_path = dataset_path

    def load(self):
        for i in tqdm(range(0, 1300)):
            try:
                with open('/data/dataset/wiki_zh/{}.txt'.format(str(i).zfill(4)), 'r') as f:
                    _lines = f.readlines()
                    for m in _lines:
                        _j = json.loads(m)
                        self._cache[int(_j["id"])] = m
            except FileNotFoundError as err:
                continue

    def __len__(self):
        return len(self._cache)
    
    def __getitem__(self, search_id=13):
        if search_id in self._cache:
            return json.loads(self._cache[search_id])
        return None

    def __iter__(self):
        self.n = 0
        return self

    def __next__(self):
        self.n += 1
        while self.n <= self.max:
            result = self[self.n]
            if result:
                return result
            else:
                self.n += 1
                continue
        raise StopIteration
        


In [5]:
ds = DataSet()

#print(ds[1625944]) #0674 锡安

#ds.load()

cnt = 0
for i in tqdm(ds):
    cnt += 1

100%|██████████| 864344/864344 [11:25<00:00, 1261.09it/s] 


In [10]:
import paddlehub as hub
module = hub.Module(name="chinese-bert-wwm")

Download https://bj.bcebos.com/paddlehub/paddlehub_dev/chinese_bert_wwm_2.0.1.tar.gz
[##################################################] 100.00%
Decompress /home/stdhi/.paddlehub/tmp/tmp6dk4mi45/chinese_bert_wwm_2.0.1.tar.gz
[##################################################] 100.00%

[2021-12-01 14:19:06,197] [    INFO] - Successfully installed chinese-bert-wwm-2.0.1
[2021-12-01 14:19:06,198] [    INFO] - Downloading http://paddlenlp.bj.bcebos.com/models/transformers/bert/bert-wwm-chinese.pdparams and saved to /home/stdhi/.paddlenlp/models/bert-wwm-chinese
[2021-12-01 14:19:06,199] [    INFO] - Downloading bert-wwm-chinese.pdparams from http://paddlenlp.bj.bcebos.com/models/transformers/bert/bert-wwm-chinese.pdparams


100%|██████████| 399504/399504 [05:11<00:00, 1282.99it/s] 


In [38]:
from itertools import chain
import paddle

SPECIAL_TOKENS = ["[CLS]", "[SEP]", "[PAD]", "[speaker1]", "[speaker2]"]

def build_input_from_segments(history, reply, tokenizer, with_eos=True):                                                                       
    """ Build a sequence of input from 3 segments: persona, history and last reply """                                                         
    bos, eos, pad, speaker1, speaker2 = tokenizer.convert_tokens_to_ids(SPECIAL_TOKENS)                                                        
    sequence = [[bos]] + history + [reply + ([eos] if with_eos else [])]                                                                       
    sequence = [sequence[0]] + [[speaker2 if i % 2 else speaker1] + s for i, s in enumerate(sequence[1:])]                                     
    instance = {}                                                                                                                              
    instance["input_ids"] = list(chain(*sequence))                                                                                             
    instance["token_type_ids"] = [bos] + [speaker2 if i % 2 else speaker1 for i, s in enumerate(sequence[1:])                                  
                                          for _ in s]                                                                                          
    return instance, sequence

def sample_sequence(history, tokenizer, model, args, current_output=None):    
    if current_output is None:                                                                                                                                                              
        current_output = []                                                                                                                                                                 
                                                                                                                                                                                            
    for i in range(28):                                                                                                                                                        
        instance, sequence = build_input_from_segments(history, current_output, tokenizer, with_eos=False)                                                                                  
        
        input_ids = [paddle.to_tensor(instance["input_ids"], dtype='int64')]
        token_type_ids = [paddle.to_tensor(instance["token_type_ids"], dtype='int64')]
        print(input_ids)
        print(token_type_ids)
        #token_type_ids = torch.tensor(instance["token_type_ids"], dtype=torch.long, device=args.device).unsqueeze(0)                                                                        
                                                                                                                                                                                            
        logits, *_ = model.forward(input_ids, token_type_ids=token_type_ids)                                                                                                                        
        logits = logits[0, -1, :] / args.temperature                                                                                                                                        
        logits = top_filtering(logits, top_k=args.top_k, top_p=args.top_p)                                                                                                                  
        probs = F.softmax(logits, dim=-1)                                                                                                                                                   
                                                                                                                                               
        prev = torch.topk(probs, 1)[1] if args.no_sample else torch.multinomial(probs, 1)                                                      
        if i < args.min_length and prev.item() in special_tokens_ids:                                                                          
            while prev.item() in special_tokens_ids:                                                                                           
                prev = torch.multinomial(probs, num_samples=1)                                                                                 
                                                                                                                                               
        if prev.item() in special_tokens_ids:                                                                                                  
            break                                                                                                                              
        current_output.append(prev.item())  
        
class Bot(object):                                                                                                                             
                                                                                                                                               
    def __init__(self, args, model, tokenizer, history):                                                                                       
        self.args = args                                                                                                                       
        self.model = model                                                                                                                     
        self.tokenizer = tokenizer                                                                                                             
        self.history = history                                                                                                                 
                                                                                                                                               
    def talk(self, inmsg):                                                                                                                     
        raw_text = inmsg                                                                                                                       
        raw_text = " ".join(list(raw_text.replace(" ", "")))                                                                                   
        while not raw_text:                                                                                                                    
            return ""                                                                                                                          
                                                                                                                                               
        #def tokenize(obj):                                                                                                                     
        #    if isinstance(obj, str):                                                                                                           
        #        return self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(obj))                                                      
        #    if isinstance(obj, dict):                                                                                                          
        #        return dict((n, self.tokenize(o)) for n, o in obj.items())                                                                     
        #    return list(tokenize(o) for o in obj)                                                                                              
                                                                                                                                               
        self.history.append(self.tokenizer.convert_tokens_to_ids(self.tokenizer.tokenize(raw_text)))                                                                                                
                                                                                                                        
        out_ids = sample_sequence(self.history, self.tokenizer, self.model, self.args)                                                     
        self.history.append(out_ids)
        self.history = self.history[-(2 * self.args.max_history + 1):]                                                                         
        out_text = self.tokenizer.decode(out_ids, skip_special_tokens=True)                                                                    
        return out_text
    
bot = Bot(None, module, module.get_tokenizer(), [])
bot.talk("你好")

[2021-12-01 20:07:05,322] [    INFO] - Already cached /home/stdhi/.paddlenlp/models/bert-wwm-chinese/bert-wwm-chinese-vocab.txt


[Tensor(shape=[5], dtype=int64, place=CPUPlace, stop_gradient=True,
       [101 , 100 , 872 , 1962, 100 ])]
[Tensor(shape=[5], dtype=int64, place=CPUPlace, stop_gradient=True,
       [101, 100, 100, 100, 100])]


AttributeError: 'bool' object has no attribute 'astype'